In [0]:
%pip install pypdf  langchain-text-splitters pdfplumber 

In [0]:
%restart_python

In [0]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import *
import uuid
from pyspark.sql import Row
import os
import pdfplumber 

In [0]:
path = "/Volumes/chatbot_catalog/acme/acme_volume/sample_business_report_tables_3_pages.pdf"

### Storing Pdf Data into table

In [0]:
pdf = PdfReader(path)

text = "\n".join(page.extract_text()  for page in pdf.pages)


document_id  =  str(uuid.uuid4())
file_name = path.split("/")[-1]
file_type = file_name.split(".")[-1]
uploaded_at  =  datetime.now(timezone.utc)
file_size_bytes = os.path.getsize(path)


row = [
    Row(
        Document_id = document_id , 
        File_Name = file_name , 
        File_Type  = file_type if file_type  else "Unknown" , 
        extracted_text = text ,
        uploaded_at = uploaded_at , 
        file_size_kb = round(file_size_bytes/1024 , 2)
    )
]

df_document = spark.createDataFrame(row)

df_document.display()
 

Document_id,File_Name,File_Type,extracted_text,uploaded_at,file_size_kb
09c31caf-29fa-4373-9b41-63fffbe2a615,sample_business_report_tables_3_pages.pdf,pdf,"Page 1 Sample Business Operations Report Purpose and instructions This three-page practice PDF contains selectable text and clearly bordered tables. Use it to practice extracting tables with Python pdfplumber, converting extracted rows into Spark DataFrames, and answering questions using structured data. Business summary Northstar Office Supplies is a fictional company. This report summarizes product sales, customer orders, inventory levels, and employee training completion for practice purposes only. Key questions to test 1. Which product generated the highest revenue? 2. Which customer order has the largest total? 3. Which products need restocking? 4. How many employees completed training? Table 1: Monthly Product Sales Product Category Units Sold Unit Price (INR) Revenue (INR) Laptop Electronics 12 50,000 600,000 Monitor Electronics 25 12,000 300,000 Keyboard Accessories 80 1,500 120,000 Mouse Accessories 100 700 70,000 Headphones Accessories 40 2,000 80,000 Webcam Electronics 18 3,000 54,000 USB Hub Accessories 35 1,200 42,000 Page 2 Sample Business Operations Report — Page 2 Table 2: Customer Orders Order ID Customer Product Quantity Total (INR) ORD-1001 Asha Traders Laptop 2 100,000 ORD-1002 Bright Office Monitor 5 60,000 ORD-1003 Asha Traders Mouse 10 7,000 ORD-1004 Metro Systems Laptop 3 150,000 ORD-1005 Bright Office Keyboard 12 18,000 ORD-1006 Metro Systems Headphones 4 8,000 ORD-1007 Asha Traders USB Hub 6 7,200 Table 3: Inventory Status Product Stock on Hand Reorder Level Status Laptop 8 5 Sufficient Monitor 6 10 Reorder Keyboard 15 20 Reorder Mouse 55 25 Sufficient Headphones 7 10 Reorder Webcam 4 8 Reorder USB Hub 20 12 Sufficient Page 3 Sample Business Operations Report — Page 3 Table 4: Employee Training Status Employee ID Department Course Status Score (%) E101 Data Engineering PySpark Basics Completed 92 E102 Analytics SQL Fundamentals In Progress 68 E103 Data Engineering Delta Lake Completed 88 E104 Operations Data Quality Not Started — E105 Analytics Power BI Completed 95 E106 Operations Safety Procedures Completed 90 E107 Data Engineering Spark Optimization In Progress 72 Table 5: Monthly Department Summary Department Employees Completed Courses Completion Rate (%) Data Engineering 3 1 33.3 Analytics 2 2 100 Operations 2 1 50 Practice tasks Try extracting all tables page by page with pdfplumber. Then convert each table into a Spark DataFrame. Practice answering: which product has the highest revenue, which inventory items need reordering, and how many employees completed their courses?",2026-10-21T04:32:54.106Z,30.57


In [0]:
table_name = "chatbot_catalog.acme.documents"

df_document.write\
    .format("delta")\
    .mode("append")\
    .option("mergeschema" , "true")\
    .saveAsTable(table_name)


In [0]:
spark.table("chatbot_catalog.acme.documents").display()

Document_id,File_Name,File_Type,extracted_text,Uploaded_at,file_size_kb
09c31caf-29fa-4373-9b41-63fffbe2a615,sample_business_report_tables_3_pages.pdf,pdf,"Page 1 Sample Business Operations Report Purpose and instructions This three-page practice PDF contains selectable text and clearly bordered tables. Use it to practice extracting tables with Python pdfplumber, converting extracted rows into Spark DataFrames, and answering questions using structured data. Business summary Northstar Office Supplies is a fictional company. This report summarizes product sales, customer orders, inventory levels, and employee training completion for practice purposes only. Key questions to test 1. Which product generated the highest revenue? 2. Which customer order has the largest total? 3. Which products need restocking? 4. How many employees completed training? Table 1: Monthly Product Sales Product Category Units Sold Unit Price (INR) Revenue (INR) Laptop Electronics 12 50,000 600,000 Monitor Electronics 25 12,000 300,000 Keyboard Accessories 80 1,500 120,000 Mouse Accessories 100 700 70,000 Headphones Accessories 40 2,000 80,000 Webcam Electronics 18 3,000 54,000 USB Hub Accessories 35 1,200 42,000 Page 2 Sample Business Operations Report — Page 2 Table 2: Customer Orders Order ID Customer Product Quantity Total (INR) ORD-1001 Asha Traders Laptop 2 100,000 ORD-1002 Bright Office Monitor 5 60,000 ORD-1003 Asha Traders Mouse 10 7,000 ORD-1004 Metro Systems Laptop 3 150,000 ORD-1005 Bright Office Keyboard 12 18,000 ORD-1006 Metro Systems Headphones 4 8,000 ORD-1007 Asha Traders USB Hub 6 7,200 Table 3: Inventory Status Product Stock on Hand Reorder Level Status Laptop 8 5 Sufficient Monitor 6 10 Reorder Keyboard 15 20 Reorder Mouse 55 25 Sufficient Headphones 7 10 Reorder Webcam 4 8 Reorder USB Hub 20 12 Sufficient Page 3 Sample Business Operations Report — Page 3 Table 4: Employee Training Status Employee ID Department Course Status Score (%) E101 Data Engineering PySpark Basics Completed 92 E102 Analytics SQL Fundamentals In Progress 68 E103 Data Engineering Delta Lake Completed 88 E104 Operations Data Quality Not Started — E105 Analytics Power BI Completed 95 E106 Operations Safety Procedures Completed 90 E107 Data Engineering Spark Optimization In Progress 72 Table 5: Monthly Department Summary Department Employees Completed Courses Completion Rate (%) Data Engineering 3 1 33.3 Analytics 2 2 100 Operations 2 1 50 Practice tasks Try extracting all tables page by page with pdfplumber. Then convert each table into a Spark DataFrame. Practice answering: which product has the highest revenue, which inventory items need reordering, and how many employees completed their courses?",2026-10-21T04:32:54.106Z,30.57



### If data is present in Table format in pdf document

In [0]:

import pdfplumber

all_tables = []

with pdfplumber.open(path) as pdf:
    for page_no, page in enumerate(pdf.pages, start=1):
        tables = page.extract_tables()
        
        for table_no , table in enumerate(tables , start = 1) :
            if table :
                all_tables.append({
                    "table_data" : table
                                   })

#### for first table 

first_table = all_tables[0]["table_data"]

header = first_table[0]
data = first_table[1:]

df_business = spark.createDataFrame(data , schema = header)

df_business.display()


Product,Category,Units Sold,Unit Price (INR),Revenue (INR)
Laptop,Electronics,12,"50,000","600,000"
Monitor,Electronics,25,"12,000","300,000"
Keyboard,Accessories,80,"1,500","120,000"
Mouse,Accessories,100,700,"70,000"
Headphones,Accessories,40,"2,000","80,000"
Webcam,Electronics,18,"3,000","54,000"
USB Hub,Accessories,35,"1,200","42,000"
